In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys

sys.path.append("../")
from tqdm import tqdm

DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_only_with_layer_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_only_with_layer_mppc_spacetime.npy"

BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_with_layer_pixel_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_with_layer_mppc_spacetime.npy"


sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)


X_pixel = torch.tensor(np.concatenate([sig_pixel_spacetime, bg_pixel_spacetime], axis=0))
X_mppc = torch.tensor(np.concatenate([sig_mppc_spacetime, bg_mppc_spacetime], axis=0))
y = torch.tensor(np.concatenate(
    [np.ones(sig_pixel_spacetime.shape[0]), np.zeros(bg_pixel_spacetime.shape[0])],
    axis=0,
))
del (
    sig_mppc_spacetime,
    sig_pixel_spacetime,
    bg_pixel_spacetime,
    bg_mppc_spacetime,
)

from sklearn.model_selection import train_test_split


X_pixel_train, X_pixel_test, X_mppc_train, X_mppc_test, y_train, y_test = (
    train_test_split(X_pixel, X_mppc, y, test_size=0.2, random_state=42, stratify=y)
)


del (
    X_pixel,
    X_mppc,
    y,
)

In [3]:
from src.torch.pre_processing import graph_batching
from importlib import reload
reload(graph_batching)

train_dataset = graph_batching.create_hetero_dataset(
    root=DATA_DIR,pixel_data=X_pixel_train,mppc_data=X_mppc_train,labels=y_train, in_memory=True)
test_dataset = graph_batching.create_hetero_dataset(
    root=DATA_DIR,pixel_data=X_pixel_test,mppc_data=X_mppc_test,labels=y_test, in_memory=True)
del X_pixel_train, X_mppc_train, y_train, X_pixel_test, X_mppc_test, y_test

Processing...


Processing detector events into in-memory graphs...
Generated 705799 graphs from 224616 events


Done!


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torch_geometric.data.storage.BaseStorage was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torch_geometric.data.storage.BaseStorage])` or the `torch.serialization.safe_globals([torch_geometric.data.storage.BaseStorage])` context manager to allowlist this global if you trust this class/function.

Check the docu

In [ ]:
test_dataset[0]

In [ ]:
from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
date

In [ ]:
from torch_geometric.loader import DataLoader
from torch_geometric import nn as PyG_nn


class HeteroEdgeClassifier(torch.nn.Module):
    """Heterogeneous Graph Edge Classifier Model
    
    This model uses heterogeneous graph convolutional layers to process graphs with multiple node and edge types. It is desgined for binary classification on the graph edges.
    
    """
    def __init__(self, hidden_channels, out_channels, num_layers, dropout):